# Z4 115M — 10-minute Kaggle profile

This is the stock **115M** baseline on the same WikiText-2 data for 600 seconds:
The stock preset keeps the 300-step warmup, aux-head compute, MTP weight 0.3,
concept top-k 8/4/2, and EMA.  It is the equal-wall control for `kaggle10m`.

The result is an early-training comparison, not a frontier claim.  See
`z3/KAGGLE_10M_ARCHITECTURE.md` for the profile and rationale.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator (T4)."
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")


In [ ]:
import os, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/nanlm_experimental")
if not (REPO / "z3" / "train_v3.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/shemshallah/nanlm_experimental.git",
                    str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
print(REPO)


In [ ]:
# Kaggle usually has torch; add only the tokenizer/data packages.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers", "numpy", "pyarrow"], check=True)


In [ ]:
# WikiText-2 raw text: small, fast, and document-separable.
import urllib.request
from pathlib import Path
Path("data").mkdir(exist_ok=True)
parts = []
for split in ("train", "valid"):
    url = f"https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/{split}.txt"
    dest = Path(f"data/wikitext_{split}.txt")
    urllib.request.urlretrieve(url, dest)
    parts.append(dest.read_text(encoding="utf-8"))
corpus = Path("data/wikitext_corpus.txt")
corpus.write_text("\n\n".join(parts), encoding="utf-8")
print(corpus, corpus.stat().st_size)


In [ ]:
# Build document-framed uint16 bins and the manifest used by the 115M trainer.
subprocess.run([
    sys.executable, "z3/build_domain_bins.py",
    "--corpus", "wikitext=data/wikitext_corpus.txt",
    "--out", "data/wikitext_bins",
    "--max-bytes-per-domain", "12000000",
    "--train-fraction", "0.98",
    "--valid-fraction", "0.01",
], check=True)


In [ ]:
# 10-minute training.  The preset caps wall time at 600 seconds.
OUT = Path("/kaggle/working/runs/z3_115m_baseline10m")
cmd = [
    sys.executable, "z3/train_v3.py",
    "--preset", "115m",
    "--corpus", "data/wikitext_bins/train.bin",
    "--tokenizer", "bin",
    "--val-corpus", "data/wikitext_bins/valid.bin",
    "--data-manifest", "data/wikitext_bins/manifest.json",
    "--out", str(OUT),
    "--max-seconds", "600",
    "--batch-size", "4",
]
print(" ".join(cmd))
try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError:
    print("batch 4 failed (likely T4 OOM); retrying with batch 2")
    subprocess.run(cmd + ["--batch-size", "2"], check=True)


In [ ]:
# Held-out per-domain evaluation.  The builder writes data/wikitext_bins/test/.
subprocess.run([
    sys.executable, "z3/eval_domains.py",
    "--checkpoint", str(OUT / "best.pt"),
    "--test-dir", "data/wikitext_bins/test",
    "--data-manifest", "data/wikitext_bins/manifest.json",
    "--tokenizer", "layered",
    "--document-framing", "--sidecar", "--require-sidecar",
    "--ctx", "512", "--batch-size", "4", "--device", "cuda",
    "--out", str(OUT / "test_domains.json"),
], check=True)


In [ ]:
import json
result = json.loads((OUT / "result.json").read_text())
test = json.loads((OUT / "test_domains.json").read_text())
print(json.dumps({
    "params": result.get("params"),
    "steps": result.get("steps_completed"),
    "tokens_seen": result.get("tokens_seen"),
    "nonfinite_skips": result.get("nonfinite_skips"),
    "best_step": (result.get("best") or {}).get("step"),
    "best_bpb": ((result.get("best") or {}).get("metrics") or {}).get("bits_per_byte"),
    "final_bpb": (result.get("final") or {}).get("bits_per_byte"),
    "macro_test_bpb": test.get("macro_bits_per_byte"),
    "feedback": (result.get("final") or {}).get("feedback"),
}, indent=2, sort_keys=True))


In [ ]:
# Optional zero-shot WikiText-2 / WikiText-103.  Keep this out of the 10-minute
# training budget; run it only if the Kaggle session still has time.
# subprocess.run([
#     sys.executable, "z3/eval_v3.py",
#     "--checkpoint", str(OUT / "best.pt"),
#     "--corpus", "wt2=eval_data/wt2_test.txt",
#     "--corpus", "wt103=eval_data/wt103_test.txt",
#     "--tokenizer", "layered", "--ctx", "512", "--batches", "8",
#     "--batch-size", "4", "--device", "cuda",
#     "--out", str(OUT / "zero_shot.json"),
# ], check=True)
